# Task 3: Vision Transformer (ViT) Fine-Tuning for Food Image Classification

This notebook implements Task 3 of the project, focusing on fine-tuning a Vision Transformer (ViT) model for food image classification using the Food-101 dataset.

We will:
1. Load and preprocess the Food-101 dataset
2. Apply data augmentation techniques
3. Fine-tune a pre-trained ViT model (`vit-base-patch16-224`)
4. Use LoRA for efficient fine-tuning
5. Evaluate using accuracy, confusion matrix, and sample predictions

## 1. Setup

Install the necessary libraries for vision tasks, including `torchvision` for data augmentation and the `datasets` library for loading the Food-101 dataset.

In [ ]:
# Install required libraries
!pip install -q transformers datasets accelerate peft torch~=2.3.0 torchvision pillow scikit-learn matplotlib seaborn

## 2. Load Dataset

We'll use the Food-101 dataset from Hugging Face Datasets, which provides an easy interface to load the data. The dataset contains 101 food categories with 1,000 images each.

In [ ]:
from datasets import load_dataset, DatasetDict

# Load the full Food-101 dataset
print("Loading full Food-101 dataset...")
full_dataset = load_dataset("food101")

# --- Create Subsets ---
# Shuffle and select a subset for training
train_subset = full_dataset['train'].shuffle(seed=42).select(range(15000))

# Shuffle the validation set and split it into validation and test sets
shuffled_val = full_dataset['validation'].shuffle(seed=42)
validation_subset = shuffled_val.select(range(5000))
test_subset = shuffled_val.select(range(5000, 10000)) # Take the next 5000 for testing

# Create a new DatasetDict with the desired splits
dataset = DatasetDict({
    'train': train_subset,
    'validation': validation_subset,
    'test': test_subset
})
# --- End Subset Creation ---

print("\nDataset structure after creating subsets:")
print(dataset)

# Show an example
print("\nExample from training set:")
print(f"Label: {dataset['train'][0]['label']}")
print(f"Image: {dataset['train'][0]['image']}")

# Get the label names
labels = dataset['train'].features['label'].names
print(f"\nNumber of categories: {len(labels)}")
print(f"First 10 categories: {labels[:10]}")

## 3. Data Preprocessing and Augmentation

We'll:
1. Load the ViT image processor to normalize images to the correct size and format
2. Apply data augmentation using torchvision transforms (RandomHorizontalFlip, RandomRotation, ColorJitter)
3. Create separate transforms for training (with augmentation) and validation (without augmentation)

In [ ]:
from transformers import AutoImageProcessor
from torchvision import transforms
import torch

model_name = "google/vit-base-patch16-224"

# Load the image processor
image_processor = AutoImageProcessor.from_pretrained(model_name)

# Define data augmentation transforms for training
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

# Define transforms for validation/testing (no augmentation)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

def train_transform_function(examples):
    """Apply training transforms to a batch of images"""
    examples['pixel_values'] = [train_transforms(img.convert("RGB")) for img in examples['image']]
    return examples

def val_transform_function(examples):
    """Apply validation/test transforms to a batch of images"""
    examples['pixel_values'] = [val_transforms(img.convert("RGB")) for img in examples['image']]
    return examples

# Apply transforms to the datasets
print("Applying transforms to training set...")
train_dataset = dataset['train'].with_transform(train_transform_function)

print("Applying transforms to validation set...")
val_dataset = dataset['validation'].with_transform(val_transform_function)

print("Applying transforms to test set...")
test_dataset = dataset['test'].with_transform(val_transform_function)


print("\nData preprocessing complete!")

## 4. Visualize Sample Images with Augmentation

Let's visualize some examples to see the effect of data augmentation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def denormalize(tensor, mean, std):
    """Denormalize a tensor image for visualization"""
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return tensor

# Get a sample batch
sample_indices = [0, 100, 200, 300]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Images from Food-101 Dataset', fontsize=16)

for idx, sample_idx in enumerate(sample_indices):
    # Original image
    original_img = dataset['train'][sample_idx]['image']
    label = labels[dataset['train'][sample_idx]['label']]
    
    axes[0, idx].imshow(original_img)
    axes[0, idx].set_title(f'Original: {label}')
    axes[0, idx].axis('off')
    
    # Augmented image
    augmented = train_transform_function({'image': [original_img], 'label': [0]})
    aug_tensor = augmented['pixel_values'][0]
    aug_img = denormalize(aug_tensor.clone(), image_processor.image_mean, image_processor.image_std)
    aug_img = aug_img.permute(1, 2, 0).numpy()
    aug_img = np.clip(aug_img, 0, 1)
    
    axes[1, idx].imshow(aug_img)
    axes[1, idx].set_title(f'Augmented: {label}')
    axes[1, idx].axis('off')

plt.tight_layout()
plt.show()

## 5. Load Model and Configure LoRA

We'll load the pre-trained ViT model and configure it with LoRA for efficient fine-tuning.

In [ ]:
from transformers import AutoModelForImageClassification
from peft import LoraConfig, get_peft_model, TaskType
import torch

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create label mappings
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for idx, label in enumerate(labels)}

# Load the pre-trained ViT model
model = AutoModelForImageClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True  # Allow changing the classification head
)

# Define LoRA configuration for ViT
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["query", "value"],  # Apply LoRA to attention query and value matrices
    lora_dropout=0.1,
    bias="none",
    modules_to_save=["classifier"],  # Also train the classification head
)

# Wrap the model with PEFT
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

print(f"\nModel loaded and configured with LoRA")

## 6. Define Evaluation Metrics

We'll compute accuracy and other metrics during training.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    """Compute accuracy and other metrics"""
    # Handle both tuple unpacking and object attribute access
    if hasattr(eval_pred, 'predictions') and hasattr(eval_pred, 'label_ids'):
        predictions = eval_pred.predictions
        labels = eval_pred.label_ids
    else:
        predictions, labels = eval_pred
    
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

In [ ]:
# Test the compute_metrics function and data loading
print("Testing data loading and metrics computation...")

# Test if we can get a batch from val_dataset
try:
    test_batch = [val_dataset[i] for i in range(4)]
    test_collated = collate_fn(test_batch)
    print(f"✓ Successfully loaded and collated a test batch")
    print(f"  Pixel values shape: {test_collated['pixel_values'].shape}")
    print(f"  Labels shape: {test_collated['labels'].shape}")
except Exception as e:
    print(f"✗ Error loading test batch: {e}")

# Test compute_metrics with dummy data
import numpy as np
dummy_predictions = np.random.rand(10, len(labels))
dummy_labels = np.random.randint(0, len(labels), 10)

class DummyPred:
    def __init__(self, predictions, label_ids):
        self.predictions = predictions
        self.label_ids = label_ids

dummy_eval_pred = DummyPred(dummy_predictions, dummy_labels)
try:
    test_metrics = compute_metrics(dummy_eval_pred)
    print(f"✓ compute_metrics working correctly: {test_metrics}")
except Exception as e:
    print(f"✗ Error in compute_metrics: {e}")

print("\nProceed to training if both tests passed.")

## 7. Fine-Tuning the Model

Now we'll set up the training process using the `Trainer` class.

In [ ]:
from transformers import TrainingArguments, Trainer
import torch

# Define training arguments
training_args = TrainingArguments(
    output_dir="vit-food101-classifier",
    num_train_epochs=5,
    per_device_train_batch_size=32,  # Adjust based on GPU memory
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=5e-4,
    fp16=True,  # Mixed precision training
    logging_dir='./logs',
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False,    # Disabled since metrics aren't being computed during training
    remove_unused_columns=False,
    save_total_limit=2,
)

# Custom data collator to handle the 'pixel_values' key
def collate_fn(batch):
    pixel_values = torch.stack([item['pixel_values'] for item in batch])
    labels = torch.tensor([item['label'] for item in batch])
    return {'pixel_values': pixel_values, 'labels': labels}

# Create the Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

# Verify the trainer is set up correctly
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Evaluation will occur at the end of each epoch")
print("\nNote: During training, evaluation metrics may not be computed.")
print("We'll run a full evaluation after training completes.\n")

# Start training
print("Starting training...")
trainer.train()

# Save the trained model
print("\nSaving the fine-tuned model...")
peft_model.save_pretrained("vit-food101-lora-adapter")
image_processor.save_pretrained("vit-food101-lora-adapter")
print("Model saved successfully!")

## 8. Evaluate on Test Set

Let's evaluate the model on the validation set and compute detailed metrics.

In [ ]:
# Evaluate the model on the validation set (as done during training)
print("Evaluating model on validation set...")
eval_results = trainer.evaluate()

print("\n=== Validation Set Evaluation Results ===")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

# Now, evaluate the final model on the held-out test set
print("\n\nEvaluating final model on the test set...")
test_results = trainer.evaluate(eval_dataset=test_dataset)

print("\n=== Test Set Evaluation Results ===")
for key, value in test_results.items():
    print(f"{key}: {value:.4f}")

## 9. Generate Predictions and Confusion Matrix

We'll generate predictions on the validation set and create a confusion matrix.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Get predictions on the test set
print("Generating predictions on the test set...")
predictions = trainer.predict(test_dataset)

# Handle predictions properly - extract logits correctly
if isinstance(predictions.predictions, tuple):
    # It's a tuple - need to find the 2D array with shape (num_samples, num_classes)
    print(f"Predictions is a tuple with {len(predictions.predictions)} elements")
    for i, item in enumerate(predictions.predictions):
        print(f"  Element {i}: shape={getattr(item, 'shape', 'N/A')}")
    
    # Find the element with shape (num_samples, num_classes)
    logits = None
    for item in predictions.predictions:
        if isinstance(item, np.ndarray) and len(item.shape) == 2:
            logits = item
            break
    
    if logits is None:
        raise ValueError("Could not find 2D logits array in predictions")
else:
    logits = predictions.predictions

print(f"Logits shape: {logits.shape}")

pred_labels = np.argmax(logits, axis=1)

# Get true labels - if not in predictions, extract from dataset
if predictions.label_ids is not None:
    true_labels = predictions.label_ids
else:
    print("Label IDs not in predictions, extracting from dataset...")
    true_labels = np.array([dataset['test'][i]['label'] for i in range(len(dataset['test']))])

print(f"Number of test samples: {len(true_labels)}")
print(f"Number of predicted labels: {len(pred_labels)}")

# Calculate confusion matrix
cm = confusion_matrix(true_labels, pred_labels)

# Plot confusion matrix
plt.figure(figsize=(20, 18))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', square=True)
plt.title('Confusion Matrix - Food-101 Classification (Test Set)', fontsize=16)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nConfusion matrix saved as 'confusion_matrix.png'")

# Print classification report for top 10 classes
print("\n=== Classification Report (First 10 Classes on Test Set) ===")
report = classification_report(true_labels, pred_labels, target_names=labels, labels=range(10))
print(report)

## 10. Sample Predictions with Visualization

Let's visualize some sample predictions to see how well the model performs.

In [ ]:
import random

# Select random samples from the test set
num_samples = 12
sample_indices = random.sample(range(len(test_dataset)), num_samples)

# Create predictions
peft_model.eval()
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Sample Predictions on Test Set', fontsize=16)

for idx, sample_idx in enumerate(sample_indices):
    row = idx // 4
    col = idx % 4
    
    # Get sample
    sample = dataset['test'][sample_idx]
    image = sample['image']
    true_label = labels[sample['label']]
    
    # Preprocess and predict
    inputs = val_transform_function({'image': [image], 'label': [0]})
    pixel_values = inputs['pixel_values'][0].unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = peft_model(pixel_values=pixel_values)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class_idx = predictions.argmax(-1).item()
        confidence = predictions[0][predicted_class_idx].item()
    
    predicted_label = labels[predicted_class_idx]
    
    # Display image
    axes[row, col].imshow(image)
    axes[row, col].axis('off')
    
    # Set title with color coding
    title_color = 'green' if predicted_label == true_label else 'red'
    axes[row, col].set_title(
        f'True: {true_label}\nPred: {predicted_label}\nConf: {confidence:.2f}',
        fontsize=10,
        color=title_color
    )

plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print("Sample predictions saved as 'sample_predictions.png'")

## 11. Top-5 Accuracy

Let's also compute the top-5 accuracy, which is a common metric for image classification tasks.

In [ ]:
# Calculate top-5 accuracy
def top_k_accuracy(predictions, labels, k=5):
    """Calculate top-k accuracy"""
    top_k_preds = np.argsort(predictions, axis=1)[:, -k:]
    correct = sum([labels[i] in top_k_preds[i] for i in range(len(labels))])
    return correct / len(labels)

top5_acc = top_k_accuracy(logits, true_labels, k=5)
print(f"\nTop-5 Accuracy: {top5_acc:.4f}")

# Find classes with best and worst performance
from sklearn.metrics import accuracy_score

per_class_accuracy = []
for class_idx in range(len(labels)):
    mask = true_labels == class_idx
    if mask.sum() > 0:
        class_acc = accuracy_score(true_labels[mask], pred_labels[mask])
        per_class_accuracy.append((labels[class_idx], class_acc))

# Sort by accuracy
per_class_accuracy.sort(key=lambda x: x[1], reverse=True)

print("\n=== Top 5 Best Performing Classes ===")
for label, acc in per_class_accuracy[:5]:
    print(f"{label}: {acc:.4f}")

print("\n=== Top 5 Worst Performing Classes ===")
for label, acc in per_class_accuracy[-5:]:
    print(f"{label}: {acc:.4f}")

## 12. (Optional) Merge LoRA with Base Model

If you want to create a standalone fine-tuned model without needing PEFT for inference, you can merge the LoRA weights with the base model.

In [ ]:
# Uncomment the following code to merge and save the model

# from peft import PeftModel

# # Reload the base model
# base_model = AutoModelForImageClassification.from_pretrained(
#     model_name,
#     num_labels=len(labels),
#     id2label=id2label,
#     label2id=label2id,
#     ignore_mismatched_sizes=True
# )

# # Load the PEFT model with LoRA adapter
# merged_model = PeftModel.from_pretrained(base_model, "vit-food101-lora-adapter")

# # Merge the weights
# merged_model = merged_model.merge_and_unload()

# # Save the merged model
# merged_model.save_pretrained("vit-food101-finetuned-merged")
# image_processor.save_pretrained("vit-food101-finetuned-merged")

# print("Merged model saved to 'vit-food101-finetuned-merged'")

## 13. Save Summary Report

Let's create a summary report of the evaluation metrics.

In [ ]:
# Calculate metrics manually from predictions
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

final_accuracy = accuracy_score(true_labels, pred_labels)
final_precision, final_recall, final_f1, _ = precision_recall_fscore_support(
    true_labels, pred_labels, average='weighted', zero_division=0
)

# Create a summary report using the calculated metrics
summary_report = f"""
=============================================================
Task 3: Vision Transformer (ViT) Food Classification Report
=============================================================

Model: {model_name}
Dataset: Food-101 (Subsets)
Training samples: {len(train_dataset)}
Validation samples: {len(val_dataset)}
Test samples: {len(test_dataset)}

Training Configuration:
- Epochs: {training_args.num_train_epochs}
- Batch size: {training_args.per_device_train_batch_size}
- Learning rate: {training_args.learning_rate}
- LoRA rank: {lora_config.r}
- Data augmentation: RandomHorizontalFlip, RandomRotation, ColorJitter

Final Test Set Evaluation Results:
- Accuracy: {final_accuracy:.4f}
- Precision: {final_precision:.4f}
- Recall: {final_recall:.4f}
- F1 Score: {final_f1:.4f}
- Top-5 Accuracy: {top5_acc:.4f}

Best Performing Classes (on Test Set):
"""

for label, acc in per_class_accuracy[:5]:
    summary_report += f"  - {label}: {acc:.4f}\n"

summary_report += "\nWorst Performing Classes (on Test Set):\n"
for label, acc in per_class_accuracy[-5:]:
    summary_report += f"  - {label}: {acc:.4f}\n"

summary_report += """
=============================================================
"""

print(summary_report)

# Save to file
with open('task3_evaluation_report.txt', 'w') as f:
    f.write(summary_report)

print("\nEvaluation report saved to 'task3_evaluation_report.txt'")